In [1]:
# REQUIRES, IMPORTS, ETC.
# =======================
# System
# ------
# import sys
import time
import importlib

# Third-party libraries
# ----------------------
import pandas as pd
import polars as pl

# MODULES
# ========
# System
# --------------  
from pathlib import Path
from typing import Dict

# Custom
# Types, Defines
# --------------
import src.config.ctypes as ctp
importlib.reload(ctp)

import src.config.globals as glb
importlib.reload(glb)
from src.config.globals import *

# Helpers functions
# -----------------
import src.lib.helpers.validations as val
importlib.reload(val)
import src.lib.helpers.console as csl
importlib.reload(csl)
import src.lib.helpers.charts as cht
importlib.reload(cht)
import src.lib.engines.bdorchestrator as bdo
importlib.reload(bdo)
from src.lib.engines.bdorchestrator import BdOrchestrator


In [2]:
# CHECK CHANGES OF IMPORT FILES
# ==============================
import importlib
importlib.reload(ctp)
importlib.reload(csl)
importlib.reload(cht)
importlib.reload(val)
importlib.reload(bdo)

print(csl.__file__)
print([name for name in dir(ctp)])
print("+ ")

print(csl.__file__)
print([name for name in dir(csl)])
print("+ ")

print(cht.__file__)
print([name for name in dir(cht)])
print("+ ")

print(val.__file__)
print([name for name in dir(val)])

print(bdo.__file__)
print([name for name in dir(bdo)])

/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/lib/helpers/console.py
['AggFunc', 'BigDataLib', 'ChartType', 'CleanStep', 'ErrorInfo', 'ErrorResponse', 'FilterOp', 'IssueAction', 'IssueType', 'Literal', 'TypeAlias', 'TypedDict', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']
+ 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/lib/helpers/console.py
['LineBreak', 'SectionBegin', 'SectionBreak', 'SectionEnd', 'SectionErrEndAndExit', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'nbsp', 'sys', 'time']
+ 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/lib/helpers/charts.py
['BarCht', 'ChartType', 'Dict', 'DifferenceCht', 'DuplicatedCht', 'HistogramCht', 'LineCht', 'PlotSalesAnalytics', 'QualityCht', 'RelationalScatterCht', 'ResidualsScatterCht', 'ScatterCht', 'SequentialSca

In [3]:
# GLOBAL VARIABLES, PATHS
# =======================
# Current file paths
# ------------------
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR.parent / "data").resolve()
ML20_DIR = (NOTEBOOK_DIR.parent / "data" / "ml-20m").resolve()

# Global variables definition for this notebook
# ---------------------------------------------
mvDataset: str = ML20_DIR / "movies.csv"
rtDataset: str = ML20_DIR / "ratings.csv"

# Pandas dataframe to hold the dataset
df: pd.DataFrame
# Algorithm result
res: Dict[str, any]

In [4]:
# LIBRARIES COMPARISON, 4.1.1 LOAD SOURCE DATASET (from CSV file)
# ===============================================================
tInit: float = time.time()
csl.SectionBegin("LIBRARIES COMPARISON, 4.1.1 LOAD SOURCE DATASET (from CSV file)\n+ Loading Movies & Ratings Datasets", 0)

try:
  # a. PANDAS::LOAD DATASET
  mvDfPd = BdOrchestrator.CleanMoviesData('pandas', 
    BdOrchestrator.LoadDataset('pandas', str(mvDataset), pPrintResult=False)['data'], 3
  )['data']
  rtDfPd: pd.DataFrame = BdOrchestrator.LoadDataset('pandas', str(rtDataset))['data']

  # b. POLARS::LOAD DATASET
  mvDfPl = BdOrchestrator.CleanMoviesData('polars', 
    BdOrchestrator.LoadDataset('polars', str(mvDataset), pPrintResult=False)['data'], 3
  )['data']
  rtDfPl: pl.DataFrame = BdOrchestrator.LoadDataset('polars', str(rtDataset))['data']

except Exception as e:
  csl.SectionErrEndAndExit(e.args[0], tInit)

+ =========================================================
+ LIBRARIES COMPARISON, 4.1.1 LOAD SOURCE DATASET (from CSV file)
+ Loading Movies & Ratings Datasets
+ ---------------------------------------------------------
Executing operation with PANDAS...
 
Movies dataset cleaned ('year' extracted) successfully.
Execution time: 0.0776 seconds
Total rows: 27255
First 3 rows of the dataset:
   movieId             title                                       genres  \
0        1         Toy Story  Adventure|Animation|Children|Comedy|Fantasy   
1        2           Jumanji                   Adventure|Children|Fantasy   
2        3  Grumpier Old Men                               Comedy|Romance   

   year  
0  1995  
1  1995  
2  1995  
+ ---------------------------------------------------------
Executing operation with PANDAS...
 
File loaded successfully.
Execution time: 7.2177 seconds
Total rows: 20000263
First 5 rows of the dataset:
   userId  movieId  rating   timestamp
0       1      

In [5]:
# LIBRARIES COMPARISON, 4.1.2a BASIC OPERATIONS: FILTERING
# =========================================================
csl.SectionBegin("LIBRARIES COMPARISON, 4.1.2a FILTERING\n+ Movies (2000 <= year) & Ratings (4 <= rating)", 0)

try:
  # a. PANDAS::FILTER & JOIN
  tInitPd: float = time.perf_counter()
  fltDfPd: pd.DataFrame = pd.merge(
    BdOrchestrator.FilterData('pandas', mvDfPd, 'year', '>=', 2000, pPrintResult=False)['data'], 
    BdOrchestrator.FilterData('pandas', rtDfPd, 'rating', '>=', 4, pPrintResult=False)['data'], 
    on='movieId', how='inner'
  )

  csl.SectionBreak(0, 0)
  print(f"Executing operation with PANDAS...")
  csl.nbsp()
  print(fltDfPd.head(3))
  csl.nbsp()
  print(f"Filtered successfully. Dataset total rows: {len(fltDfPd)}")
  print(f"Execution time: {time.perf_counter() - tInitPd:.4f} seconds")

  # b. POLARS::FILTER & JOIN
  tInitPl: float = time.perf_counter()
  fltDfPl: pl.DataFrame = BdOrchestrator.FilterData('polars', mvDfPl, 'year', '>=', 2000, pPrintResult=False)['data'].join(
     BdOrchestrator.FilterData('polars', rtDfPl, 'rating', '>=', 4, pPrintResult=False)['data'], on='movieId', how='inner'
  )

  csl.SectionBreak(0, 0)
  print(f"Executing operation with POLARS...")
  csl.nbsp()
  print(fltDfPl.head(3))
  csl.nbsp()
  print(f"Filtered successfully. Dataset total rows: {len(fltDfPl)}")
  print(f"Execution time: {time.perf_counter() - tInitPl:.4f} seconds")

except Exception as e:
  csl.SectionErrEndAndExit(e.args[0], tInit)

csl.SectionEnd(0)

+ =========================================================
+ LIBRARIES COMPARISON, 4.1.2a FILTERING
+ Movies (2000 <= year) & Ratings (4 <= rating)
+ ---------------------------------------------------------
Executing operation with PANDAS...
 
   movieId       title       genres  year  userId  rating   timestamp
0     2769  Yards, The  Crime|Drama  2000     156     4.0  1046895842
1     2769  Yards, The  Crime|Drama  2000    1358     5.0   973554543
2     2769  Yards, The  Crime|Drama  2000    2196     4.0   996462288
 
Filtered successfully. Dataset total rows: 2500965
Execution time: 2.1252 seconds
+ ---------------------------------------------------------
Executing operation with POLARS...
 
shape: (3, 7)
┌─────────┬─────────────────────┬────────────────────────────┬──────┬────────┬────────┬────────────┐
│ movieId ┆ title               ┆ genres                     ┆ year ┆ userId ┆ rating ┆ timestamp  │
│ ---     ┆ ---                 ┆ ---                        ┆ ---  ┆ ---    

In [6]:
# LIBRARIES COMPARISON, 4.1.2b BASIC OPERATIONS: SORTING
# ======================================================
tInit: float = time.time()
sortCols: list[str] = ['year', 'rating', 'movieId', 'userId']
sortDirs: list[bool] = [False, False, True, True]

csl.SectionBegin( f"LIBRARIES COMPARISON, 4.1.3 SORTING\n+ Sort by: {sortCols} | Ascending: {sortDirs}", 0)

try:

  # a. PANDAS::SORT DATA
  srtDfPd: pd.DataFrame = BdOrchestrator.SortData(
    'pandas',
    fltDfPd,
    sortCols,
    sortDirs,
    pPreviewRows=3
  )['data']

  # b. POLARS::SORT DATA
  srtDfPl: pl.DataFrame = BdOrchestrator.SortData(
    'polars',
    fltDfPl,
    sortCols,
    sortDirs,
    pPreviewRows=3
  )['data']

except Exception as e:
  csl.SectionErrEndAndExit(e.args[0], tInit)

csl.SectionEnd(0)

+ =========================================================
+ LIBRARIES COMPARISON, 4.1.3 SORTING
+ Sort by: ['year', 'rating', 'movieId', 'userId'] | Ascending: [False, False, True, True]
+ ---------------------------------------------------------
Executing operation with PANDAS...
 
Data sorted by ['year', 'rating', 'movieId', 'userId'] successfully.
Execution time: 0.8253 seconds
Total rows: 2500965
First 3 rows of the dataset:
   movieId              title                   genres  year  userId  rating  \
0   113345  Jupiter Ascending  Action|Adventure|Sci-Fi  2015   19089     5.0   
1   113345  Jupiter Ascending  Action|Adventure|Sci-Fi  2015   41852     5.0   
2   113345  Jupiter Ascending  Action|Adventure|Sci-Fi  2015   98139     5.0   

    timestamp  
0  1425219438  
1  1424739035  
2  1427469523  
+ ---------------------------------------------------------
Executing operation with POLARS...
 
Data sorted by ['year', 'rating', 'movieId', 'userId'] successfully.
Execution time

In [7]:
# LIBRARIES COMPARISON, 4.1.2c BASIC OPERATIONS: AGGREGATION
# =========================================================
tInit: float = time.time()
groupCols: list[str] = ['year', 'movieId', 'title']
targetCol: str = 'rating'
aggOperation: str = 'mean'
aliasCol: str = 'avg_rating'

csl.SectionBegin(f"LIBRARIES COMPARISON, 4.1.2c AGGREGATION\n+ Group by: {groupCols}\n+ Operation: {aggOperation}('{targetCol}') as '{aliasCol}'", 0)

try:
  # a. PANDAS::AGGREGATE DATA
  aggDfPd: pd.DataFrame = BdOrchestrator.AggregateData(
    'pandas',
    srtDfPd,
    groupCols,
    targetCol,
    aggOperation,
    aliasCol,
    pPreviewRows=3
  )['data']

  # b. POLARS::AGGREGATE DATA
  aggDfPl: pl.DataFrame = BdOrchestrator.AggregateData(
    'polars',
    srtDfPl,
    groupCols,
    targetCol,
    aggOperation,
    aliasCol,
    pPreviewRows=3
  )['data']

except Exception as e:
  csl.SectionErrEndAndExit(e.args[0], tInit)

csl.SectionEnd(0)

+ =========================================================
+ LIBRARIES COMPARISON, 4.1.2c AGGREGATION
+ Group by: ['year', 'movieId', 'title']
+ Operation: mean('rating') as 'avg_rating'
+ ---------------------------------------------------------
Executing operation with PANDAS...
 
Data aggregated by ['year', 'movieId', 'title'] (mean of 'rating') successfully.
Execution time: 0.3471 seconds
Total rows: 9735
First 3 rows of the dataset:
   year  movieId        title  avg_rating
0  2000     2769   Yards, The    4.218750
1  2000     3177  Next Friday    4.247292
2  2000     3190    Supernova    4.153409
+ ---------------------------------------------------------
Executing operation with POLARS...
 
Data aggregated by ['year', 'movieId', 'title'] (mean of 'rating') successfully.
Execution time: 0.1608 seconds
Total rows: 9735
First 3 rows of the dataset:
shape: (3, 4)
┌──────┬─────────┬─────────────┬────────────┐
│ year ┆ movieId ┆ title       ┆ avg_rating │
│ ---  ┆ ---     ┆ ---      